# 뉴스 기사 요약 프로젝트 통합 정리

환경 점검부터 최종 평가까지의 핵심 코드·저장 결과·해석을 한 파일로 정리했다. 작성 기준: 2026-09-21.

**실행하지 않은 설명용 통합본**이다. 실행 번호와 출력은 비워 두었고, 아래 결과표는 기존 산출물에서 읽어 Markdown으로 넣었다. 코드 예제는 `RUN_EXAMPLES=False`로 차단되어 있다. 원본 실험이나 결과 파일을 덮어쓰지 않는다.

흐름: 00 환경 → 01 EDA → 02 토큰화·임베딩 → 03 베이스라인 → 04 정제·그룹 분할 → 05-1 A0~A3 → 05-2 설정 비교 → 05-3 A4~A8 → 05-4 디코딩 → 05-5 test → 05-6 기초 오류 분석.

In [ ]:
from pathlib import Path
import json, re, random
from collections import Counter
import pandas as pd
from IPython.display import display, Image
PROJECT = Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
RUN_EXAMPLES = False  # 핵심 코드 열람용; 전체 재현은 원본 노트북 사용
SEED = 42
random.seed(SEED)
def saved_json(relative_path):
    return json.loads((PROJECT / relative_path).read_text(encoding='utf-8-sig'))


## 00. 환경과 입력·정답
데이터는 총 27,400건(train 22,194 / validation 2,466 / test 2,740). 2022년 7월 1~10일 경제·IT과학 기사다. 모델 입력은 document, 학습 정답은 summary이며 제목·언론사·날짜·링크는 입력하지 않는다. 사전학습 자체는 외부에서 완료된 모델을 사용했고, 이 프로젝트에서는 파인튜닝을 수행했다.

In [ ]:
display(saved_json('results/00_environment_data_kobart/report.json')['environment'])


## 01. EDA: 문제 발견
길이·분야 분포, 결측·중복, 반복 요약, 숫자 불일치, 본문 대비 요약 길이를 검사했다. 자동 플래그는 오류 확정이 아닌 검토 후보다. 짧은 사진기사와 긴 방송 대본은 길이 분포에 영향을 준다.

In [ ]:
def noise_features(document, summary):
    d, s = str(document), str(summary)
    tokens = re.findall(r'[가-힣A-Za-z0-9]+(?:[.,]\d+)?', s.lower())
    counts = Counter(tuple(tokens[i:i+4]) for i in range(len(tokens)-3))
    overlap = sum(w in set(d.split()) for w in s.split()) / max(len(s.split()), 1)
    ratio = len(s) / max(len(d), 1)
    return dict(repeat_4gram=max(counts.values(), default=0)>=3,
                low_overlap=overlap<0.3, extreme_ratio=ratio<0.02 or ratio>0.8,
                summary_longer=len(s)>len(d), short_document=len(d.split())<30)


### 원본 노이즈 후보
저장된 결과: `review_data/04-1_raw_candidates/00_export_summary.csv`

| file | rows |
| --- | --- |
| 01_repeat_4gram.csv | 1376 |
| 02_low_overlap.csv | 6 |
| 03_extreme_ratio.csv | 2823 |
| 04_summary_longer.csv | 2132 |
| 05_number_mismatch.csv | 777 |
| 06_short_document.csv | 1514 |
| 07_url.csv | 83 |
| 08_boilerplate.csv | 53 |
| 09_replacement_char.csv | 4 |
| 10_exact_document_summary_duplicate.csv | 3785 |
| 11_shared_summary_different_document.csv | 100 |

In [ ]:
display(pd.read_csv(PROJECT / 'review_data/04-1_raw_candidates/00_export_summary.csv'))


## 02. 토큰화·임베딩 및 모델 선택
KoBART의 encoder-decoder 구조는 본문을 조건으로 요약을 생성하기에 적합하다. KoGPT2의 decoder-only 구조와는 이론·토큰화만 비교했으며, 동일 조건 파인튜닝 대결은 아직 하지 않았다. BPE가 보편적으로 WordPiece보다 우월하다는 결론이 아니라, 선택한 KoBART 가중치와 기존 토크나이저의 호환성을 유지했다.

### 같은 문장 토큰화 비교
출처: `02_tokenize_embedding.ipynb`. 아래 코드는 기존 구현의 핵심 발췌이며 재실행용 전체 파이프라인은 원본 노트북을 사용한다. 기본값에서는 실행하지 않는다.

In [ ]:
if RUN_EXAMPLES:
    from transformers import AutoTokenizer
    specs = {'KoBART': 'gogamza/kobart-base-v2', 'KoGPT2': 'skt/kogpt2-base-v2',
             'KoELECTRA': 'monologg/koelectra-base-v3-discriminator'}
    text = '정부가 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대한다'
    for name, model_id in specs.items():
        tok = AutoTokenizer.from_pretrained(model_id)
        print(name, tok.tokenize(text))


### 토큰 길이 비교
저장된 결과: `results/02_tokenize_embedding/token_length_comparison.csv`

| tokenizer | mean | over_512_pct | over_1024_pct |
| --- | --- | --- | --- |
| KoBART_BPE | 418.1155 | 27.5210 | 4.1363 |
| KoGPT2_BPE | 422.6594 | 28.0752 | 4.2489 |
| KoELECTRA_WordPiece | 501.9608 | 37.8345 | 7.6237 |

In [ ]:
display(pd.read_csv(PROJECT / 'results/02_tokenize_embedding/token_length_comparison.csv'))


예시 문장의 특수 토큰 제외 토큰 수는 KoBART 17, KoGPT2 19, KoELECTRA 20이다. KoELECTRA의 22는 특수 토큰을 포함한 수다. 서로 다른 집계 기준을 혼동하지 않는다.

### Word2Vec의 고정 임베딩
출처: `02_tokenize_embedding.ipynb`. 아래 코드는 기존 구현의 핵심 발췌이며 재실행용 전체 파이프라인은 원본 노트북을 사용한다. 기본값에서는 실행하지 않는다.

In [ ]:
if RUN_EXAMPLES:
    from gensim.models import Word2Vec
    from konlpy.tag import Okt
    okt = Okt()
    # train_df는 원본 train 22,194건을 불러온 DataFrame
    corpus = [okt.morphs(x, stem=True) for x in train_df.document.astype(str)]
    w2v = Word2Vec(corpus, vector_size=100, window=5, min_count=5,
                   workers=1, sg=1, seed=42, epochs=5)
    print(w2v.wv.most_similar('수출', topn=10))


### 문맥 벡터와 shape 흐름
출처: `02_tokenize_embedding.ipynb, sample inputs / kobart_model은 앞 단계에서 준비`. 아래 코드는 기존 구현의 핵심 발췌이며 재실행용 전체 파이프라인은 원본 노트북을 사용한다. 기본값에서는 실행하지 않는다.

In [ ]:
if RUN_EXAMPLES:
    sample_text = train_df.loc[0, "document"]
    sample_inputs = kobart_tokenizer(
        sample_text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
        return_token_type_ids=False,
    )
    sample_inputs = {key: value.to(device) for key, value in sample_inputs.items()}

    decoder_start_token_id = kobart_model.config.decoder_start_token_id
    if decoder_start_token_id is None:
        decoder_start_token_id = (
            kobart_tokenizer.bos_token_id
            if kobart_tokenizer.bos_token_id is not None
            else kobart_tokenizer.eos_token_id
        )
    decoder_input_ids = torch.tensor([[decoder_start_token_id]], device=device)

    with torch.no_grad():
        input_embeddings = kobart_model.model.shared(sample_inputs["input_ids"])
        encoder_output = kobart_model.model.encoder(**sample_inputs).last_hidden_state
        model_output = kobart_model(
            **sample_inputs,
            decoder_input_ids=decoder_input_ids,
        )

    shape_log = {
        "input_ids": list(sample_inputs["input_ids"].shape),
        "attention_mask": list(sample_inputs["attention_mask"].shape),
        "token_embedding": list(input_embeddings.shape),
        "encoder_last_hidden_state": list(encoder_output.shape),
        "decoder_input_ids": list(decoder_input_ids.shape),
        "decoder_logits": list(model_output.logits.shape),
        "hidden_size": kobart_model.config.d_model,
        "vocab_size": kobart_model.config.vocab_size,
    }


In [ ]:
display(saved_json('results/02_tokenize_embedding/kobart_shape_log.json'))
display(pd.read_csv(PROJECT/'results/02_tokenize_embedding/embedding_context_comparison.csv'))


동일한 ‘수출’ 토큰의 입력 벡터 코사인 유사도는 약 1.000, 문맥을 반영한 encoder 벡터는 약 0.356이었다. 이는 문맥 표현이 달라짐을 보여주며 성능 우열 점수는 아니다.

## 03. 베이스라인
03-1: KoBART base를 파인튜닝 없이 적용. 03-2: raw에 가까운 데이터로 파인튜닝. 03-3: 앞 50/80어절을 사용하는 Lead. 세 방식은 원래 test 2,740건을 사용했다. 이후 정제 test 2,564건의 점수와 직접 같은 조건으로 비교하지 않는다.

In [ ]:
def lead_summary(document, n=50):
    return ' '.join(str(document).split()[:n])
display(saved_json('results/03-1_baseline_no_summary/zero_shot_metrics.json'))
display(saved_json('results/03-2_baseline_raw_data/test_metrics.json'))
display(pd.read_csv(PROJECT/'results/03-3_baseline_lead/lead_metrics.csv'))


ROUGE-L: zero-shot 17.12 / raw 파인튜닝 32.18 / Lead-50 39.26 / Lead-80 39.72. Lead가 강한 이유로 정답의 추출적 성격과 기사 앞부분 편향이 관찰됐다. raw 학습은 epoch 2 이후 validation loss가 상승했다.

## 04. 공통 전처리와 누수 방지
04-1 후보 분류 → 04-2 반복 행 제외·본문 정리 → 04-3 완전 중복 제거·동일 document/summary 그룹 → 04-4 남은 후보 재분류. 반복 구절 축약을 검토했으나 완전한 복원 보장이 어려워 행 제외를 채택했다. 기자·언론사 이름은 유지한다. 숫자 수동 검토는 탐색했지만 현재 A 계열 학습에는 채택하지 않았다.

### 실제 문자열 정리 규칙
출처: `04-2_clean_dataset.ipynb`. 아래 코드는 기존 구현의 핵심 발췌이며 재실행용 전체 파이프라인은 원본 노트북을 사용한다. 기본값에서는 실행하지 않는다.

In [ ]:
if RUN_EXAMPLES:
    word_pattern=re.compile(r"[가-힣A-Za-z0-9]+(?:[.,]\d+)?")
    def max_4gram(text):
        tokens=word_pattern.findall(str(text).lower())
        counts=Counter(tuple(tokens[i:i+4]) for i in range(len(tokens)-3))
        return max(counts.values(),default=0)
    URL=re.compile(r"(?i)\b(?:https?://|www\.)[^\s<>()\[\]{}]+")
    BOILERPLATES=[
     re.compile(r"국내에서만\s*사용가능\.?\s*재배포\s*금지\.?\s*DB\s*금지\.?\s*For\s*Use\s*Only\s*in\s*the\s*Republic\s*of\s*Korea\.?\s*(?:No\s*)?Redistribution\s*Prohibited",re.I),
     re.compile(r"무단\s*(?:전재|복제|배포)\s*(?:및\s*)?재배포\s*금지\.?",re.I),
     re.compile(r"재배포\s*금지\.?",re.I),
     re.compile(r"인터뷰\s*자료의\s*저작권은\s*[^.]{0,120}(?:있습니다|있다)\.?",re.I),
     re.compile(r"(?:전문\s*게재나?\s*)?인터뷰\s*인용\s*보도\s*시\s*[^.]{0,180}(?:바랍니다|바란다)\.?",re.I),
     re.compile(r"인용\s*보도\s*시\s*출처를\s*[^.]{0,120}(?:바랍니다|바란다)\.?",re.I),
     re.compile(r"아래\s*텍스트는\s*실제\s*방송\s*내용과\s*차이가\s*있을\s*수\s*있으니\s*[^.]{0,180}(?:바랍니다|바란다)\.?",re.I)]
    def clean_document(text):
        original=str(text); cleaned,url_n=URL.subn("",original); boiler_n=0
        for pattern in BOILERPLATES:
            cleaned,n=pattern.subn("",cleaned); boiler_n+=n
        replacement_n=cleaned.count("�"); cleaned=cleaned.replace("�"," ")
        cleaned=re.sub(r"[ \t]+"," ",cleaned); cleaned=re.sub(r" *\n *","\n",cleaned).strip()
        return cleaned,url_n,boiler_n,replacement_n


### 반복 제거와 본문 정리
저장된 결과: `review_data/04-2_cleaning/cleaning_summary.csv`

| split | original_rows | repeat_excluded | clean_rows | documents_changed |
| --- | --- | --- | --- | --- |
| train | 22194 | 1121 | 21073 | 104 |
| validation | 2466 | 121 | 2345 | 9 |
| test | 2740 | 134 | 2606 | 16 |

In [ ]:
display(pd.read_csv(PROJECT / 'review_data/04-2_cleaning/cleaning_summary.csv'))


### 완전 중복 제거와 연결 그룹
출처: `04-3_dedup_group_split.ipynb; INPUT은 04-2_cleaned 폴더`. 아래 코드는 기존 구현의 핵심 발췌이며 재실행용 전체 파이프라인은 원본 노트북을 사용한다. 기본값에서는 실행하지 않는다.

In [ ]:
if RUN_EXAMPLES:
    splits=["train","validation","test"]
    df=pd.concat([pd.read_csv(INPUT/f"{s}.csv") for s in splits],ignore_index=True)
    assert df.row_id.is_unique
    priority={"test":0,"validation":1,"train":2}
    ordered=df.assign(_priority=df.split.map(priority),_position=np.arange(len(df))).sort_values(["_priority","source_index","_position"])
    ordered["keep_exact"]=~ordered.duplicated(["document","summary"],keep="first")
    df=df.merge(ordered[["row_id","keep_exact"]],on="row_id",validate="one_to_one")
    exact_excluded=df[~df.keep_exact].copy(); work=df[df.keep_exact].copy().reset_index(drop=True)

    # 같은 document 또는 같은 summary를 공유하면 같은 연결 요소로 묶습니다.
    parent=list(range(len(work)))
    def find(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]; x=parent[x]
        return x
    def union(a,b):
        ra,rb=find(a),find(b)
        if ra!=rb: parent[rb]=ra
    for column in ["document","summary"]:
        for indices in work.groupby(column,sort=False).indices.values():
            indices=list(indices)
            for index in indices[1:]: union(indices[0],index)
    work["group_id"]=[find(i) for i in range(len(work))]
    group_splits=work.groupby("group_id").split.agg(set)
    def destination(values):
        if "test" in values: return "test"
        if "validation" in values: return "validation"
        return "train"
    destinations=group_splits.map(destination)
    work["assigned_split"]=work.group_id.map(destinations)
    work["keep_group"]=~((work.assigned_split=="test")&(work.split!="test"))
    work["split_action"]="kept"
    work.loc[~work.keep_group,"split_action"]="excluded_test_overlap"
    moved=work.keep_group & work.split.ne(work.assigned_split)
    work.loc[moved,"split_action"]="moved_to_"+work.loc[moved,"assigned_split"]


### 그룹 처리 후 데이터
저장된 결과: `review_data/04-3_dedup_group_split/split_summary.csv`

| split | input_rows | exact_duplicates_excluded | final_rows | test_overlap_excluded |
| --- | --- | --- | --- | --- |
| train | 21073 | 1831 | 19236 | 4 |
| validation | 2345 | 106 | 2240 | 1 |
| test | 2606 | 42 | 2564 | 0 |

In [ ]:
display(pd.read_csv(PROJECT / 'review_data/04-3_dedup_group_split/split_summary.csv'))


원본 test에서 반복·중복 행을 제외한 test를 고정했다. 동일 document 또는 summary로 test와 연결된 비-test 행은 제외하고, train/validation 사이 연결 그룹은 validation으로 모았다. 이는 정확히 동일한 텍스트 기준이며 모든 유사 사건 기사 누수까지 제거했다는 뜻은 아니다.

### 04-4 잔여 후보
저장된 결과: `review_data/04-4_final_quality_review/00_export_summary.csv`

| file | rows |
| --- | --- |
| 01_repeat_4gram_remaining.csv | 0 |
| 02_low_overlap.csv | 0 |
| 03_extreme_ratio.csv | 1024 |
| 04_summary_longer.csv | 610 |
| 05_short_document.csv | 486 |
| 06_short_and_summary_longer.csv | 414 |
| 07_number_mismatch.csv | 643 |
| 08_remaining_boilerplate.csv | 2 |

In [ ]:
display(pd.read_csv(PROJECT / 'review_data/04-4_final_quality_review/00_export_summary.csv'))


## 05. 공통 학습 핵심
입력 512토큰, 정답 256토큰, pad 정답은 -100으로 loss에서 제외한다. loss는 다음 정답 토큰 예측의 손실이며 ROUGE와 동일한 지표가 아니다. 아래는 Dataset·학습·validation·생성 함수의 기존 구현 발췌다.

### 학습·평가 함수
출처: `05-3_noise_ablation.ipynb; torch/np, DataLoader 및 tokenizer/model 설정은 원본 참고`. 아래 코드는 기존 구현의 핵심 발췌이며 재실행용 전체 파이프라인은 원본 노트북을 사용한다. 기본값에서는 실행하지 않는다.

In [ ]:
if RUN_EXAMPLES:
    def reset_seed():
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    def make_dataset(frame):
        source = tokenizer(
            text=frame["document"].astype(str).tolist(),
            padding="max_length", truncation=True, max_length=MAX_SOURCE_LENGTH,
            return_token_type_ids=False, return_tensors="pt",
        )
        target = tokenizer(
            text_target=frame["summary"].astype(str).tolist(),
            padding="max_length", truncation=True, max_length=MAX_TARGET_LENGTH,
            return_token_type_ids=False, return_tensors="pt",
        )
        labels = target["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100
        return TensorDataset(source["input_ids"], source["attention_mask"], labels)

    def make_loader(dataset, train_mode):
        sampler = RandomSampler(dataset) if train_mode else SequentialSampler(dataset)
        return DataLoader(dataset, sampler=sampler, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)

    def train_one_epoch(model, optimizer, dataloader):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(dataloader, start=1):
            input_ids, attention_mask, labels = [tensor.to(device) for tensor in batch]
            optimizer.zero_grad()
            loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
            loss.backward()
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            if step % 500 == 0:
                print(f"  step {step:,}/{len(dataloader):,} | loss {total_loss / step:.4f}")
        return total_loss / len(dataloader)

    def evaluate_loss(model, dataloader):
        model.eval()
        total_loss = 0.0
        with torch.no_grad():
            for batch in dataloader:
                input_ids, attention_mask, labels = [tensor.to(device) for tensor in batch]
                total_loss += model(
                    input_ids=input_ids, attention_mask=attention_mask, labels=labels
                ).loss.item()
        return total_loss / len(dataloader)

    def generate_predictions(model, dataloader):
        model.eval()
        predictions = []
        with torch.no_grad():
            for step, batch in enumerate(dataloader, start=1):
                input_ids, attention_mask, _ = [tensor.to(device) for tensor in batch]
                generated_ids = model.generate(
                    input_ids=input_ids, attention_mask=attention_mask,
                    max_length=MAX_TARGET_LENGTH, num_beams=1, do_sample=False,
                )
                predictions.extend(tokenizer.batch_decode(
                    generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
                ))
                if step % 200 == 0:
                    print(f"  generation {step:,}/{len(dataloader):,}")
        return [prediction.strip() for prediction in predictions]


### best checkpoint와 early stopping
출처: `05-3_noise_ablation.ipynb의 핵심 흐름 축약`. 아래 코드는 기존 구현의 핵심 발췌이며 재실행용 전체 파이프라인은 원본 노트북을 사용한다. 기본값에서는 실행하지 않는다.

In [ ]:
if RUN_EXAMPLES:
    best_loss, stale = float('inf'), 0
    for epoch in range(1, 6):
        train_loss = train_one_epoch(model, optimizer, train_loader)
        val_loss = evaluate_loss(model, validation_loader)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'validation_loss': val_loss})
        if val_loss < best_loss:
            best_loss, stale = val_loss, 0
            torch.save(model.state_dict(), model_path)
        else:
            stale += 1
            if stale >= 1:
                break
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))


### 05-1 A0~A3 초기 실험
저장된 결과: `results/05-1_noise_ablation/experiment_results.csv`

| experiment | train_rows | validation_rows | best_epoch | best_validation_loss | rougeL |
| --- | --- | --- | --- | --- | --- |
| A0 | 20839 | 2811 | 1 | 0.5147 | 29.4609 |
| A1 | 20058 | 2605 | 1 | 0.5109 | 37.3015 |
| A2 | 20058 | 2605 | 2 | 0.5102 | 30.0245 |
| A3 | 19236 | 2240 | 2 | 0.5057 | 38.4167 |

In [ ]:
display(pd.read_csv(PROJECT / 'results/05-1_noise_ablation/experiment_results.csv'))


A0 원본에 가까운 train, A1 반복 오류 제외, A2 본문 정리 추가, A3 중복·그룹 처리 추가. 학습률 3e-5, 5 epoch. 표의 ROUGE-L은 test 점수다. validation 구성도 달라 val loss만으로 전처리 효과를 단정할 수 없다. 단일 seed 결과로 통계적 유의성을 검증하지 않았다.

### 05-2 A3 하이퍼파라미터
저장된 결과: `results/05-2_hyperparameter_tuning/all_trial_results.csv`

| trial | effective_batch_size | best_epoch | best_validation_loss | rougeL |
| --- | --- | --- | --- | --- |
| lr_1em05_len_512 | 2 | 2 | 0.4682 | 37.5722 |
| lr_2em05_len_512 | 2 | 2 | 0.4861 | 36.7677 |
| lr_1em05_len_1024 | 2 | 2 | 0.4902 | 34.6564 |

In [ ]:
display(pd.read_csv(PROJECT / 'results/05-2_hyperparameter_tuning/all_trial_results.csv'))


1e-5 / 512를 선택했다. 1024는 물리 batch 1 × accumulation 2로 유효 batch 2를 유지했지만 성능은 낮았다. 기존 3e-5의 test ROUGE와 이 validation ROUGE를 직접 비교하지 않는다.

### 05-3 독립 노이즈 제거
저장된 결과: `results/05-3_noise_ablation/experiment_results.csv`

| experiment | removed_train_rows | best_epoch | best_validation_loss | rougeL |
| --- | --- | --- | --- | --- |
| A3 | 0 | 2 | 0.4682 | 37.5722 |
| A4 | 756 | 3 | 0.4777 | 41.9678 |
| A5 | 450 | 3 | 0.4731 | 39.4278 |
| A6 | 368 | 2 | 0.4738 | 32.3703 |
| A7 | 314 | 3 | 0.4815 | 35.9173 |
| A8 | 514 | 2 | 0.4761 | 35.1965 |

In [ ]:
display(pd.read_csv(PROJECT / 'results/05-3_noise_ablation/experiment_results.csv'))


### A4~A8 train 구성
출처: `05-3_noise_ablation.ipynb`. 아래 코드는 기존 구현의 핵심 발췌이며 재실행용 전체 파이프라인은 원본 노트북을 사용한다. 기본값에서는 실행하지 않는다.

In [ ]:
if RUN_EXAMPLES:
    candidate_files = {'A4':'03_extreme_ratio.csv', 'A5':'04_summary_longer.csv',
                       'A6':'05_short_document.csv', 'A7':'06_short_and_summary_longer.csv',
                       'A8':'07_number_mismatch.csv'}
    base_train = pd.read_csv(PROJECT/'processed_data/04-3_final/train.csv')
    validation = pd.read_csv(PROJECT/'processed_data/04-3_final/validation.csv')
    for experiment, filename in candidate_files.items():
        candidates = pd.read_csv(PROJECT/'review_data/04-4_final_quality_review'/filename)
        train = base_train.loc[~base_train.row_id.isin(candidates.row_id)].copy()
        print(experiment, len(train), len(validation))


각 실험은 A3에서 독립적으로 train 후보만 제거했다. A4 756건, A5 450건, A6 368건, A7 314건, A8 514건 제외. A4의 validation ROUGE-L이 가장 높았다. 숫자 후보 전부 제외한 A8은 기준보다 낮았으므로 채택하지 않았다. best epoch 이후 loss 상승이 남아 과적합이 해소됐다고 표현하지 않는다.

### 05-4 디코딩
저장된 결과: `results/05-4_decoding_validation/decoding_results.csv`

| strategy | num_beams | rouge1 | rouge2 | rougeL |
| --- | --- | --- | --- | --- |
| beam_5 | 5 | 49.0211 | 41.4913 | 44.9142 |
| beam_3 | 3 | 48.2627 | 40.8088 | 44.1263 |
| greedy | 1 | 45.8705 | 38.4971 | 41.9741 |

In [ ]:
display(pd.read_csv(PROJECT / 'results/05-4_decoding_validation/decoding_results.csv'))


### 같은 모델의 디코딩 비교
출처: `decoding_validation.py; A4 best checkpoint, 동일 validation`. 아래 코드는 기존 구현의 핵심 발췌이며 재실행용 전체 파이프라인은 원본 노트북을 사용한다. 기본값에서는 실행하지 않는다.

In [ ]:
if RUN_EXAMPLES:
    model.eval()
    for beams in [1, 3, 5]:
        with torch.inference_mode():
            generated = model.generate(**inputs, max_length=256,
                                       num_beams=beams, do_sample=False)
        prediction = tokenizer.batch_decode(generated, skip_special_tokens=True,
                                           clean_up_tokenization_spaces=False)


## 06. 최종 test 평가
validation에서 A4 + beam 5를 선택한 뒤 test 2,564건의 요약을 생성했다. 저장 예측으로 평가 패키지 설치·파일 잠금·평가 모델 설정 문제를 해결하며 점수를 재계산했다. 추가 파인튜닝은 하지 않았다.

### 한국어 ROUGE와 BERTScore
출처: `final_test_evaluation.py`. 아래 코드는 기존 구현의 핵심 발췌이며 재실행용 전체 파이프라인은 원본 노트북을 사용한다. 기본값에서는 실행하지 않는다.

In [ ]:
if RUN_EXAMPLES:
    import evaluate
    rouge = evaluate.load('rouge').compute(
        predictions=predictions, references=references,
        tokenizer=lambda text: text.split(), use_stemmer=False)
    bert = evaluate.load('bertscore').compute(
        predictions=predictions, references=references, model_type='xlm-roberta-base',
        lang='ko', device='cuda', batch_size=16, rescale_with_baseline=False)


| rouge1 | rouge2 | rougeL | bertscore_precision | bertscore_recall | bertscore_f1 |
| --- | --- | --- | --- | --- | --- |
| 49.4328 | 41.9345 | 45.6207 | 90.8948 | 92.7816 | 91.7755 |

BERTScore F1 91.78은 원점수 0.9178에 100을 곱한 값이며 정답률 91.78%가 아니다. evaluator는 XLM-R base, baseline rescaling 미사용이다. 초기 KoBART 실험에서 test를 이미 확인했으므로 완전히 미관측 test를 단 한 번만 사용했다는 주장은 하지 않는다. 저장 generation_seconds는 재시도 실행 구간을 포함한 값으로 실제 최초 생성 시간으로 쓰지 않는다.

## 07. 기초 오류 분석과 시각화
기사별 공백 ROUGE F1, 분야별·길이별 집계, 하위 50건 및 무작위 50건을 저장했다. 이는 오류 유형 수동 판정 완료를 의미하지 않는다. 두 표본은 중복될 수 있어 합친 뒤 row_id 중복을 제거해야 한다.

### 기사별 평가와 검토 표본
출처: `error_analysis.py의 핵심과 검토용 표 결합 예시`. 아래 코드는 기존 구현의 핵심 발췌이며 재실행용 전체 파이프라인은 원본 노트북을 사용한다. 기본값에서는 실행하지 않는다.

In [ ]:
if RUN_EXAMPLES:
    from rouge_score.rouge_scorer import RougeScorer
    class SpaceTokenizer:
        def tokenize(self, text): return str(text).split()
    scorer = RougeScorer(['rouge1','rouge2','rougeL'], tokenizer=SpaceTokenizer())
    frame['rougeL'] = [scorer.score(r,p)['rougeL'].fmeasure*100
                       for r,p in zip(frame.summary,frame.prediction)]
    review = pd.concat([frame.nsmallest(50,'rougeL'),frame.sample(50,random_state=42)])
    review = review.drop_duplicates('row_id').assign(오류유형='', 메모='')


### 본문 길이별 성능
저장된 결과: `results/05-6_error_analysis/length_bin_metrics.csv`

| length_bin | rows | rougeL |
| --- | --- | --- |
| ≤100 | 438 | 49.2979 |
| 101–250 | 1199 | 46.0371 |
| 251–500 | 757 | 43.8177 |
| 501–1000 | 154 | 42.8557 |
| >1000 | 16 | 27.9587 |

In [ ]:
display(pd.read_csv(PROJECT / 'results/05-6_error_analysis/length_bin_metrics.csv'))


In [ ]:
display(Image(filename=str(PROJECT / 'results/05-6_error_analysis/figures/01_rougeL_distribution.png')))


In [ ]:
display(Image(filename=str(PROJECT / 'results/05-6_error_analysis/figures/02_category_rougeL.png')))


In [ ]:
display(Image(filename=str(PROJECT / 'results/05-6_error_analysis/figures/03_length_bin_rougeL.png')))


In [ ]:
display(Image(filename=str(PROJECT / 'results/05-6_error_analysis/figures/04_prediction_reference_length.png')))


## 08. 완료 범위와 한계
완료: EDA, 토큰화·임베딩, 3종 베이스라인, 자동 전처리, 정확 중복 그룹 처리, A0~A8, 학습률·입력 길이, greedy/beam, 최종 점수, 기초 분석.

미완료: KoGPT2 파인튜닝 구조 비교, split 비율 비교, train 양 10/50/100%, batch 독립 비교, 명시적 epoch 비교표, 반복억제·sampling·길이 조절, 형태소 ROUGE 비교, 통합 experiment_log.csv, 사람의 오류 유형 판정과 지정 그래프, 발표자료. 현재 PDF는 진행 결과 보고서이며 가이드 전체 완료 보고서는 아니다.

현재 네 그래프는 성능 분포·분야·길이·생성 길이 분석이다. 가이드 지정의 오류 유형 막대·길이×오류 유형 히트맵·길이 vs ROUGE 산점도·개선 전후 오류 비교는 별도로 필요하다.